# 06 - GNN Graph Diagnostics

Inspect the persisted DB-first GNN state without retraining.

This notebook:
- loads the active DB-first leaf graph from `.vault-exec/vault.kv`
- loads the persisted GNN params already produced by live training
- recomputes GNN embeddings in-memory for diagnostics only
- reports graph-level and embedding-drift metrics

It does not persist params and it does not train the model.


In [1]:
import {
  getActiveTrainingBuildId,
  listActiveToolLeafEdgesNext,
  listActiveToolLeafNodes,
} from "../src/training-data/rebuild.ts";
import {
  buildToolLeafGraphNodes,
  buildToolLeafSeedEmbeddings,
} from "../src/training-data/model-inputs.ts";
import { openVaultStore } from "../src/db/index.ts";
import { gnnForward } from "../src/gnn/application/forward.ts";
import { DEFAULT_GNN_CONFIG } from "../src/gnn/domain/types.ts";
import { deserializeGnnParams } from "../src/gnn/infrastructure/params-codec.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

function cosine(left: number[], right: number[]) {
  let dot = 0;
  let leftNorm = 0;
  let rightNorm = 0;
  for (let index = 0; index < left.length; index++) {
    dot += left[index] * right[index];
    leftNorm += left[index] * left[index];
    rightNorm += right[index] * right[index];
  }
  return dot / (Math.sqrt(leftNorm) * Math.sqrt(rightNorm) + 1e-9);
}

function mean(values: number[]) {
  if (values.length === 0) return 0;
  return values.reduce((sum, value) => sum + value, 0) / values.length;
}

function median(values: number[]) {
  if (values.length === 0) return 0;
  const sorted = [...values].sort((left, right) => left - right);
  const middle = Math.floor(sorted.length / 2);
  return sorted.length % 2 === 0
    ? (sorted[middle - 1] + sorted[middle]) / 2
    : sorted[middle];
}

const kv = await Deno.openKv(DB_PATH);
const buildId = await getActiveTrainingBuildId(kv);
if (!buildId) {
  throw new Error(`No active DB-first build found in ${DB_PATH}. Run 'deno task cli init ${VAULT_PATH}' first.`);
}
const nodeRows = await listActiveToolLeafNodes(kv);
const edgeRows = await listActiveToolLeafEdgesNext(kv);
kv.close();

const store = await openVaultStore(DB_PATH);
const storedParams = await store.getGnnParams();
store.close();

if (!storedParams) {
  throw new Error("No persisted GNN params found. Run init/sync and wait for live training to finish.");
}

const params = await deserializeGnnParams(storedParams.params);
const seedEmbeddings = buildToolLeafSeedEmbeddings(nodeRows, DEFAULT_GNN_CONFIG.embDim);
const graphNodes = buildToolLeafGraphNodes(nodeRows, edgeRows, seedEmbeddings);
const gnnEmbeddings = gnnForward(graphNodes, params, DEFAULT_GNN_CONFIG);

const outgoingWeightByLeaf = new Map<string, number>();
const incomingWeightByLeaf = new Map<string, number>();
for (const edge of edgeRows) {
  outgoingWeightByLeaf.set(edge.fromLeaf, (outgoingWeightByLeaf.get(edge.fromLeaf) ?? 0) + edge.weight);
  incomingWeightByLeaf.set(edge.toLeaf, (incomingWeightByLeaf.get(edge.toLeaf) ?? 0) + edge.weight);
}

const rows = nodeRows.map((row) => {
  const seed = seedEmbeddings.get(row.leafKey);
  const gnn = gnnEmbeddings.get(row.leafKey);
  return {
    leafKey: row.leafKey,
    level: row.level,
    totalOccurrences: row.totalOccurrences,
    uniqueSessions: row.uniqueSessions,
    uniqueAgents: row.uniqueAgents,
    outgoingWeight: outgoingWeightByLeaf.get(row.leafKey) ?? 0,
    incomingWeight: incomingWeightByLeaf.get(row.leafKey) ?? 0,
    cosineSeedToGnn: seed && gnn ? cosine(seed, gnn) : null,
  };
}).sort((left, right) => right.totalOccurrences - left.totalOccurrences || left.leafKey.localeCompare(right.leafKey));

const cosineValues = rows.flatMap((row) => row.cosineSeedToGnn === null ? [] : [row.cosineSeedToGnn]);
const summary = {
  buildId,
  gnnParamsEpoch: storedParams.epoch,
  gnnParamsAccuracy: storedParams.accuracy,
  embDim: DEFAULT_GNN_CONFIG.embDim,
  nodeCount: nodeRows.length,
  edgeCount: edgeRows.length,
  meanCosineSeedToGnn: mean(cosineValues),
  medianCosineSeedToGnn: median(cosineValues),
  maxOutgoingWeight: Math.max(0, ...rows.map((row) => row.outgoingWeight)),
  maxIncomingWeight: Math.max(0, ...rows.map((row) => row.incomingWeight)),
};

console.log(`Active build: ${summary.buildId}`);
console.log(`Leaf nodes: ${summary.nodeCount}`);
console.log(`Leaf transitions: ${summary.edgeCount}`);
console.log(`Persisted GNN params epoch: ${summary.gnnParamsEpoch}`);
console.log(`Mean cosine(seed, gnn): ${summary.meanCosineSeedToGnn.toFixed(4)}`);
console.log(`Median cosine(seed, gnn): ${summary.medianCosineSeedToGnn.toFixed(4)}`);
console.table(rows.slice(0, 20).map((row) => ({
  ...row,
  cosineSeedToGnn: row.cosineSeedToGnn?.toFixed(4) ?? "n/a",
})));

Active build: 2b103f4f-a717-44f2-83ef-305eb0149166


Leaf nodes: 107


Leaf transitions: 994


Persisted GNN params epoch: 0


Mean cosine(seed, gnn): 0.5116


Median cosine(seed, gnn): 0.4724


┌───────┬─────────────────────────────────────┬───────┬──────────────────┬────────────────┬──────────────┬────────────────┬────────────────┬─────────────────┐
│ (idx) │ leafKey                             │ level │ totalOccurrences │ uniqueSessions │ uniqueAgents │ outgoingWeight │ incomingWeight │ cosineSeedToGnn │
├───────┼─────────────────────────────────────┼───────┼──────────────────┼────────────────┼──────────────┼────────────────┼────────────────┼─────────────────┤
│     0 │ "tool.exec.other_shell"             │     2 │             3372 │            177 │            3 │           3332 │           3331 │ "0.0522"        │
│     1 │ "tool.exec.inspect_fs_text"         │     2 │              962 │             70 │            3 │            951 │            959 │ "0.0561"        │
│     2 │ "tool.exec.network_http"            │     2 │              655 │            135 │            3 │            644 │            553 │ "0.0913"        │
│     3 │ "tool.process.poll"                 

In [2]:
const degreeSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: "Most active leaf nodes in the active GNN build",
  width: 760,
  height: 320,
  data: { values: rows.slice(0, 30) },
  mark: "bar",
  encoding: {
    x: { field: "totalOccurrences", type: "quantitative", title: "Occurrences" },
    y: { field: "leafKey", type: "nominal", sort: "-x", title: "Leaf key" },
    color: { field: "level", type: "nominal", title: "Level" },
    tooltip: [
      { field: "leafKey" },
      { field: "totalOccurrences" },
      { field: "outgoingWeight" },
      { field: "incomingWeight" },
      { field: "cosineSeedToGnn", format: ".4f" }
    ]
  }
};

const cosineSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: "Seed-to-GNN cosine distribution",
  width: 760,
  height: 220,
  data: {
    values: rows.filter((row) => row.cosineSeedToGnn !== null),
  },
  mark: "bar",
  encoding: {
    x: { field: "cosineSeedToGnn", type: "quantitative", bin: true, title: "Cosine(seed, gnn)" },
    y: { aggregate: "count", type: "quantitative", title: "Leaf count" },
    color: { field: "level", type: "nominal", title: "Level" }
  }
};

({
  vconcat: [degreeSpec, cosineSpec],
  config: { view: { stroke: null } },
});

{
  vconcat: [
    {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      title: "Most active leaf nodes in the active GNN build",
      width: 760,
      height: 320,
      data: {
        values: [
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object],
          [Object], [Object], [Object]
        ]
      },
      mark: "bar",
      encoding: {
        x: {
          field: "totalOccurrences",
          type: "quantitative",
          title: "Occurrences"
        },
        y: {
          field: "leafKey",
          type: "nominal",
          sort: "-x",
          title: "Leaf key"
        },
        color: { field: "level", type: "nominal", title: "Level" },
   